# Week 7: NumPy, the arrays behind every raster

**GEOG0111 Scientific Computing**
MSc Environmental Modelling · MSc Remote Sensing and Environmental Mapping · UCL Geography

Last week you read Sentinel-2 bands, built Earth Engine composites and calculated NDVI. Every one of those rasters, once it is in Python, is a **NumPy array**: `rioxarray` and `xarray` wrap NumPy arrays with coordinates, `pandas` columns are NumPy arrays, an image you load with `matplotlib` is a NumPy array (Week 5), and a GeoTIFF you export from Earth Engine opens as one.

This week we open the box. You will learn to work with the arrays directly: check their shape and data type, select pixels and windows, summarise along time or space, mask clouds, classify, reshape and combine arrays of different shapes. At the end you put it all together on a **year of real Sentinel-2 images** over Richmond Park, using only NumPy after the data are read.

These skills are exactly what your **Part B** processing section needs: short functions that take arrays in and return arrays out.

**How to use this notebook:** it is a worked walk-through. Run the cells in order, read the explanations, and stop at the **Questions**. Think about your answer first (many ask you to *predict* a shape or a value), then click **Show answer**. Change numbers and rerun cells: with NumPy, the fastest way to learn is to print `.shape` and look.

## What you will learn

1. why arrays beat lists for numerical work, and how much faster vectorised code is;
2. the anatomy of an array: `shape`, `ndim`, `size`, `dtype`, `nbytes`, and how big a Sentinel-2 tile is in memory;
3. the **data-type traps** that silently corrupt EO results (integer overflow, truncation, no NaN in integers);
4. creating arrays, reproducible random numbers and **coordinate grids**;
5. indexing and slicing 2-D and 3-D rasters, band order, sampling at field plots, and **views vs copies**;
6. statistics along an **axis**: per-pixel maps vs per-date time series;
7. **masks**: boolean logic, `np.where`, `np.select`, `np.isin`, NaN-aware statistics and **bit-packed quality flags**;
8. reshaping: raster ↔ table, block averaging, finding where the maximum is;
9. **broadcasting**: per-band scaling, anomalies, distance grids and model grids, without loops;
10. a complete NumPy workflow on a real Sentinel-2 time series, saved with `np.savez_compressed`.

---
# 0. Setup

Only section 10 needs extra packages (to read Sentinel-2 from the cloud, as in Week 6). Everything else uses NumPy and matplotlib, which Colab already has.

In [ ]:
# Needed only for section 10 (about 30 s in Colab)
%pip install -q rioxarray pystac-client

In [ ]:
from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

DATA_DIR = Path('data')
(DATA_DIR / 'processed').mkdir(parents=True, exist_ok=True)

print('NumPy version:', np.__version__)

---
# 1. Why arrays?

A Python **list** can hold anything (numbers, text, other lists) and each element is a separate Python object. A NumPy **array** holds values of **one data type**, packed next to each other in memory. That allows two things: arithmetic on whole arrays at once (**vectorisation**), and doing it in fast compiled code rather than a Python loop.

In [ ]:
python_list = [1, 2, 3, 4, 5]
numpy_array = np.array([1, 2, 3, 4, 5])

print('list * 2 :', python_list * 2)      # repeats the list
print('array * 2:', numpy_array * 2)      # multiplies each element

How much faster? Below we calculate NDVI for a 500 × 500 pixel image (250,000 pixels, a tiny fraction of one Sentinel-2 tile) twice: with a Python loop over every pixel, and with one vectorised line.

In [ ]:
rng = np.random.default_rng(42)                 # reproducible random numbers (section 4)
red = rng.uniform(0.02, 0.15, size=(500, 500))
nir = rng.uniform(0.20, 0.50, size=(500, 500))

# 1) Python loop over every pixel
start = time.perf_counter()
ndvi_loop = np.zeros_like(red)
for i in range(red.shape[0]):
    for j in range(red.shape[1]):
        ndvi_loop[i, j] = (nir[i, j] - red[i, j]) / (nir[i, j] + red[i, j])
t_loop = time.perf_counter() - start

# 2) Vectorised: the same formula applied to whole arrays
start = time.perf_counter()
ndvi_vec = (nir - red) / (nir + red)
t_vec = time.perf_counter() - start

print(f'loop:       {t_loop:.3f} s')
print(f'vectorised: {t_vec:.4f} s  ->  about {t_loop / t_vec:,.0f} times faster')
print('same result:', np.allclose(ndvi_loop, ndvi_vec))

**Question:** A full Sentinel-2 tile has 10980 × 10980 pixels at 10 m. Roughly how long would the loop take for one tile, and why does this matter for a time series?

<details><summary><b>Show answer</b></summary>

The tile has about 120 million pixels, roughly 480 times more than our 250,000, so multiply your loop time by ~480: typically one to several minutes per tile, and hours for a year of images (~70 dates). The vectorised version takes about a second per tile. (The exact speed-up depends on the computer, typically tens to hundreds of times.) In EO, *how* you write the arithmetic decides whether an analysis is practical at all. **Rule: if you are looping over pixels, there is almost always a NumPy way to avoid it.**

</details>

---
# 2. The anatomy of an array

Five attributes tell you almost everything about an array. **Print them every time you load data**: most bugs show up as an unexpected shape or data type.

| Attribute | Meaning |
|---|---|
| `.shape` | length of each axis, e.g. `(bands, rows, columns)` |
| `.ndim` | number of axes (dimensions) |
| `.size` | total number of elements |
| `.dtype` | data type of every element |
| `.nbytes` | memory used, in bytes |

In [ ]:
image = np.zeros((4, 300, 400), dtype=np.uint16)   # 4 bands, 300 rows, 400 columns

print('shape :', image.shape)
print('ndim  :', image.ndim)
print('size  :', image.size)
print('dtype :', image.dtype)
print('nbytes:', image.nbytes, 'bytes =', image.nbytes / 1e6, 'MB')

Memory is `size × bytes per element`. Sentinel-2 stores reflectance as **`uint16`** (2 bytes, whole numbers 0–65535). Converting to floating point for calculations multiplies the memory:

In [ ]:
n_pixels = 10980 * 10980          # one Sentinel-2 tile, one 10 m band

for dtype in [np.uint16, np.float32, np.float64]:
    bytes_per_pixel = np.dtype(dtype).itemsize
    print(f'{np.dtype(dtype).name:8s} {bytes_per_pixel} bytes/pixel -> {n_pixels * bytes_per_pixel / 1e6:7.0f} MB per band')

**Question:** A free Colab session has about 12 GB of RAM. What happens if you load all 13 Sentinel-2 bands of one tile at 10 m as `float64`? What are two ways around it?

<details><summary><b>Show answer</b></summary>

13 × 964 MB ≈ **12.5 GB**: more than the whole session, so Colab crashes ("Your session crashed after using all available RAM"). Ways around it: (1) read only the **window** you need (Week 6: `clip_box`) and the **bands** you need; (2) use a coarser **overview** level; (3) use `float32` instead of `float64` (half the memory, and more than enough precision for reflectance); (4) let Earth Engine do the computation on the server. This is why we always check `.shape`, `.dtype` and `.nbytes` before doing anything big.

</details>

---
# 3. Data-type traps

NumPy will usually **not** warn you when the data type changes your numbers. These three traps are common in EO work.

## 3.1 Integer overflow: unsigned integers cannot be negative

Sentinel-2 and Landsat bands are delivered as unsigned integers (`uint16`). Subtracting two `uint16` arrays where the result is negative **wraps around** to a large positive number:

In [ ]:
red_dn = np.array([1200, 800, 3000], dtype=np.uint16)    # stored digital numbers (DN)
nir_dn = np.array([3500, 600, 2800], dtype=np.uint16)    # pixels 2 and 3 have nir < red (water, bare soil)

print('nir - red as uint16: ', nir_dn - red_dn)
print('NDVI from uint16:    ', (nir_dn - red_dn) / (nir_dn + red_dn))

nir_f, red_f = nir_dn.astype(np.float32), red_dn.astype(np.float32)
print('NDVI from float32:   ', (nir_f - red_f) / (nir_f + red_f))

−200 became 65336, so NDVI for water became 46.7, an impossible value (NDVI is between −1 and 1). The fix: **convert to float (`.astype(np.float32)`) before any arithmetic.** Converting DN to reflectance (DN × scale + offset) does this for you, which is one more reason to always convert first.

## 3.2 Truncation: an integer array cannot store decimals

In [ ]:
classes = np.zeros(3, dtype=np.int32)
classes[0] = 0.73          # silently truncated
classes[1] = 2.99          # truncated, not rounded
print(classes)

## 3.3 No NaN in integer arrays

`NaN` ("not a number") is how we mark missing data (clouds, outside the study area) in floating-point arrays. Integers have no NaN, which is why integer rasters use a **nodata value** instead (Sentinel-2: 0; many products: −9999 or 65535).

In [ ]:
dn = np.array([0, 1520, 1610], dtype=np.uint16)    # 0 = nodata in Sentinel-2
try:
    dn[0] = np.nan
except ValueError as e:
    print('ValueError:', e)

# The usual pattern: convert to float, then replace the nodata value with NaN
refl = dn.astype(np.float32)
refl[refl == 0] = np.nan
print(refl * 0.0001)

**Question:** Why do data providers store reflectance as `uint16` DN (e.g. 1520) instead of floats (0.152)?

<details><summary><b>Show answer</b></summary>

**Storage and transfer.** `uint16` uses 2 bytes per pixel, `float32` uses 4 and `float64` uses 8, so integers halve or quarter the size of petabyte-scale archives. With a scale factor of 0.0001, the precision (0.0001 reflectance) is far finer than the sensor's noise, so nothing useful is lost. The price is that **you** must apply the scale and offset (Week 6) and handle the nodata value before analysis.

</details>

---
# 4. Creating arrays

| Function | Use in EO work |
|---|---|
| `np.array([...])` | from a list (small, hand-typed data) |
| `np.zeros`, `np.ones`, `np.full(shape, value)` | empty outputs, masks, a raster filled with a nodata value |
| `np.zeros_like(a)`, `np.full_like(a, value)` | a new array with the **same shape and dtype** as an existing raster |
| `np.arange(start, stop, step)` | integer sequences (years, band numbers, indices); `stop` is excluded |
| `np.linspace(start, stop, n)` | exactly `n` evenly spaced values, `stop` included |
| `np.random.default_rng(seed)` | reproducible random numbers (simulations, noise, sampling) |

In [ ]:
template = np.zeros((3, 4), dtype=np.float32)
nodata_raster = np.full_like(template, np.nan)          # same shape and dtype, filled with NaN
print(nodata_raster)

print('\nYears:      ', np.arange(2017, 2025))
print('Thresholds: ', np.linspace(0, 1, 6))

rng = np.random.default_rng(seed=0)                     # the same seed -> the same numbers every run
print('Random:     ', rng.normal(loc=20, scale=3, size=4).round(2))

**Question:** Why might `np.arange(0, 1, 0.1)` be a bad idea, and what should you use instead?

<details><summary><b>Show answer</b></summary>

With a **decimal step**, rounding errors in floating-point arithmetic can make `arange` produce one value more or fewer than you expect, depending on the numbers. `np.linspace(0, 0.9, 10)` states the number of values directly and is safe. Use `arange` for integers, `linspace` for decimals.

</details>

## 4.1 Coordinate grids

A raster is a grid of pixels, and each pixel centre has map coordinates. If we know the upper-left corner and the pixel size (the **geotransform**, Week 5 section 8), the coordinates of every pixel centre follow. `np.meshgrid` turns a 1-D list of x values and a 1-D list of y values into two 2-D grids:

In [ ]:
x_ul, y_ul, res = 520000, 175000, 20        # upper-left corner (British National Grid, m) and 20 m pixels
n_rows, n_cols = 3, 4

x = x_ul + (np.arange(n_cols) + 0.5) * res  # pixel-centre eastings
y = y_ul - (np.arange(n_rows) + 0.5) * res  # northings DEcrease downwards (row 0 = north)

X, Y = np.meshgrid(x, y)
print('X (easting):\n', X)
print('\nY (northing):\n', Y)

**Question:** Why do we *subtract* for `y`, and why `+ 0.5`?

<details><summary><b>Show answer</b></summary>

Row 0 of an image is the **top (north)** edge, and northing decreases as the row number increases, so we subtract. `+ 0.5` moves from the pixel's corner to its **centre**. Mixing up corners and centres shifts every coordinate by half a pixel (10 m here), a classic source of small misalignments between datasets.

</details>

---
# 5. Indexing and slicing rasters

## 5.1 Single pixels and windows

For a 2-D array, `a[row, col]`. Slices use `start:stop:step` with `stop` **excluded**, and `:` means "everything on this axis". Let's make a small simulated 3-band image (bands, rows, columns):

In [ ]:
rng = np.random.default_rng(1)
img = rng.integers(0, 100, size=(3, 6, 8))     # 3 bands x 6 rows x 8 columns
print('shape:', img.shape)

print('band 0, row 2, col 5:  ', img[0, 2, 5])
print('all bands at that pixel:', img[:, 2, 5])             # a pixel's "spectrum"
print('band 1, rows 1-3, cols 2-5:\n', img[1, 1:4, 2:6])     # a window
print('last row of band 2:    ', img[2, -1, :])              # negative index counts from the end

**Question:** Predict the shapes of `img[0]`, `img[:, 0]`, `img[:, :, ::2]` and `img[:, 1:4, 1:4]`, then check them with `.shape`.

<details><summary><b>Show answer</b></summary>

`img[0]` → `(6, 8)`: one band. `img[:, 0]` → `(3, 8)`: the first row of every band. `img[:, :, ::2]` → `(3, 6, 4)`: every second column. `img[:, 1:4, 1:4]` → `(3, 3, 3)`. An integer index **removes** that axis; a slice **keeps** it (even a slice of length 1, e.g. `img[0:1]` → `(1, 6, 8)`).

</details>

## 5.2 Band order: band-first vs band-last

Geospatial libraries (`rasterio`, `rioxarray`, Earth Engine exports) give you **band-first** arrays: `(bands, rows, cols)`. `matplotlib.imshow` wants an RGB image **band-last**: `(rows, cols, 3)`. `np.moveaxis` swaps between them without copying data:

In [ ]:
rgb_band_first = rng.uniform(0, 0.3, size=(3, 60, 80))        # e.g. red, green, blue reflectance
rgb_band_last = np.moveaxis(rgb_band_first, 0, -1)            # move axis 0 to the end
print(rgb_band_first.shape, '->', rgb_band_last.shape)

fig, ax = plt.subplots(figsize=(4, 3))
ax.imshow(np.clip(rgb_band_last / 0.3, 0, 1))                 # stretch to 0-1 for display (Week 5)
ax.set_title('Band-last array shown as RGB')
ax.axis('off')
plt.show()

## 5.3 Sampling at field plots: indexing with arrays

You often need raster values at a set of locations (field plots, weather stations, flux towers). If you know their rows and columns, pass them as two arrays: NumPy pairs them up element by element.

In [ ]:
ndvi = np.round(rng.uniform(0.1, 0.9, size=(6, 8)), 2)

plot_rows = np.array([0, 2, 4, 5])
plot_cols = np.array([1, 3, 6, 7])

print('NDVI at the 4 plots:', ndvi[plot_rows, plot_cols])     # pixels (0,1), (2,3), (4,6), (5,7)

# The same indexing can WRITE values, e.g. a mask marking where the plots are
plot_mask = np.zeros(ndvi.shape, dtype=np.int8)
plot_mask[plot_rows, plot_cols] = 1
print(plot_mask)

## 5.4 Views and copies: a silent trap

A slice is a **view** of the original data, not a copy. Changing the view changes the original raster:

In [ ]:
dem = np.arange(20, dtype=np.float32).reshape(4, 5)
window = dem[:2, :2]          # a view
window[:] = -9999             # "just set my window to nodata"...
print(dem)                    # ...the original DEM changed too

dem = np.arange(20, dtype=np.float32).reshape(4, 5)
window = dem[:2, :2].copy()   # an independent copy
window[:] = -9999
print(dem)                    # unchanged

**Question:** Why does NumPy return views for slices, when it can cause surprises like this?

<details><summary><b>Show answer</b></summary>

**Speed and memory.** A view is just a new way of looking at the same memory, so slicing a 1 GB raster costs nothing. Copying would double the memory every time. The rule: slices (`a[1:3]`, `a[::2]`) give views; indexing with arrays or masks (`a[[0, 2]]`, `a[a > 0]`) gives copies. Use `.copy()` whenever you plan to modify a subset and keep the original intact.

</details>

---
# 6. Statistics along an axis

`np.mean`, `np.median`, `np.std`, `np.min`, `np.max`, `np.sum` and `np.percentile` work on the whole array, or **along an axis**. This is the key to time series of images.

Imagine a stack of 12 monthly land surface temperature (LST) images, shape `(time, rows, cols)`:

In [ ]:
rng = np.random.default_rng(7)
months = np.arange(1, 13)
seasonal = 12 + 10 * np.sin((months - 4) / 12 * 2 * np.pi)       # warm in summer, cold in winter (°C)
urban = np.zeros((5, 6))
urban[1:4, 2:5] = 4                                               # a 4 °C warmer "city" in the middle
lst = seasonal[:, np.newaxis, np.newaxis] + urban + rng.normal(0, 0.5, size=(12, 5, 6))
# (the [:, np.newaxis, np.newaxis] trick is explained in section 9)

print('stack shape:            ', lst.shape)
print('mean over time (axis=0):', lst.mean(axis=0).shape, '-> one map')
print('mean over space (1, 2): ', lst.mean(axis=(1, 2)).shape, '-> one value per month')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

im = axes[0].imshow(lst.mean(axis=0), cmap='inferno')
axes[0].set_title('Annual mean LST per pixel (axis=0)')
fig.colorbar(im, ax=axes[0], label='°C')

axes[1].plot(months, lst.mean(axis=(1, 2)), marker='o')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Mean LST (°C)')
axes[1].set_title('Spatial mean per month (axis=(1, 2))')
plt.tight_layout()
plt.show()

**The rule:** the axis you name is the axis that **disappears**. Averaging over `axis=0` (time) leaves a map; averaging over axes 1 and 2 (space) leaves a time series.

**Question:** What shape do `lst.max(axis=0)`, `lst.std(axis=(1, 2))` and `lst.mean(axis=0, keepdims=True)` have? What is `keepdims` for?

<details><summary><b>Show answer</b></summary>

`(5, 6)`, `(12,)` and `(1, 5, 6)`. `keepdims=True` keeps the reduced axis with length 1, so the result can be combined directly with the original stack, e.g. `lst - lst.mean(axis=0, keepdims=True)` gives the anomaly of every month from the pixel's annual mean (section 9 explains why that works).

</details>

In [ ]:
# argmax along an axis: WHICH month is warmest, for every pixel
warmest_month = lst.argmax(axis=0) + 1          # +1 because index 0 = January
print(warmest_month)

## 6.1 Real data: objects launched into space each year

Our World in Data publishes the number of objects (satellites, probes, crewed spacecraft, space-station modules) launched into space each year since 1957, per country and for the **World**. It is derived from the United Nations Office for Outer Space Affairs (UNOOSA) *Online Index of Objects Launched into Outer Space*. We read the CSV with `pandas` (Week 4), keep the World rows, and move the numbers into NumPy arrays with `.to_numpy()`: from then on it is all NumPy.

In [ ]:
import pandas as pd

OWID_URL = 'https://ourworldindata.org/grapher/yearly-number-of-objects-launched-into-outer-space.csv'
try:
    df = pd.read_csv(OWID_URL, storage_options={'User-Agent': 'Mozilla/5.0 (GEOG0111 teaching notebook)'})
    world = df[df['Entity'] == 'World'].sort_values('Year')
    years = world['Year'].to_numpy()
    launched = world.iloc[:, 3].to_numpy()                 # 4th column: the annual number of objects
    print('Downloaded from Our World in Data;', df['Entity'].nunique(), 'entities in the file')
except Exception as e:
    print(f'Download failed ({type(e).__name__}); using a copy of the World values (1957-2025, downloaded Nov 2026).')
    years = np.arange(1957, 2026)
    launched = np.array([2, 8, 14, 20, 38, 77, 71, 107, 161, 145, 159, 140, 138, 130, 156, 134, 138, 128, 158, 158,
                         137, 165, 124, 130, 158, 145, 154, 163, 165, 134, 135, 145, 139, 170, 137, 130, 108, 123,
                         105, 100, 152, 157, 129, 121, 86, 96, 88, 74, 72, 95, 111, 109, 125, 121, 129, 134, 210,
                         241, 222, 221, 456, 452, 586, 1274, 1814, 2479, 2903, 2849, 4510])

print('years:   ', years.shape, years.dtype, years[0], '...', years[-1])
print('launched:', launched.shape, launched.dtype, '| total', launched.sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, scale in zip(axes, ['linear', 'log']):
    ax.bar(years, launched, color='tab:blue', width=0.9)
    ax.set_yscale(scale)
    ax.set_xlabel('Year')
    ax.set_ylabel('Objects launched')
    ax.set_title(f'Objects launched into space per year, World ({scale} scale)')
plt.tight_layout()
plt.show()

The same tools as before, now on a real 1-D series: `argmax`, masks, cumulative sums and differences.

In [ ]:
i_max = launched.argmax()
print(f'Record year: {years[i_max]} with {launched[i_max]} objects')

busy = launched >= 1000
print('Years with 1000 or more objects:', years[busy])

since_2020 = years >= 2020
print(f'Share of all objects ever launched that were launched since 2020: {launched[since_2020].sum() / launched.sum():.0%}')

cumulative = np.cumsum(launched)                        # running total
change = np.diff(launched)                              # year-on-year change: one value shorter!
print(f'Total by 2000: {cumulative[years == 2000][0]}, by {years[-1]}: {cumulative[-1]}')
print(f'Largest increase from one year to the next: +{change.max()} (from {years[change.argmax()]} to {years[change.argmax() + 1]})')

### Back to 2-D: totals per decade with `reshape`

Take the six complete decades 1960–2019 (60 years), reshape them into a `(6, 10)` table, one **row** per decade, and sum along `axis=1`:

In [ ]:
in_range = (years >= 1960) & (years <= 2019)
by_decade = launched[in_range].reshape(6, 10)           # rows = decades, columns = year within the decade
print(by_decade)

decade_totals = by_decade.sum(axis=1)                   # sum across each row -> one value per decade
for start, total in zip(range(1960, 2020, 10), decade_totals):
    print(f'{start}s: {total:5d}')
print(f'2020-{years[-1]}: {launched[years >= 2020].sum():5d}  (only {np.sum(years >= 2020)} years!)')

**Question:** Why is the year-on-year `change` array one element shorter than `launched`, and which year does `change[0]` belong to?

<details><summary><b>Show answer</b></summary>

`np.diff` subtracts neighbours: `launched[1] - launched[0]`, `launched[2] - launched[1]`, ... With 69 years there are only 68 differences. `change[0]` is the change **from 1957 to 1958**, so to label it you need `years[1:]` (or `years[:-1]` if you label by the starting year). Off-by-one errors like this are very common with `diff`, `cumsum` and moving windows: always check the lengths.

</details>

**Question:** What does the log-scale plot show that the linear plot hides? What drove the jump after 2019?

<details><summary><b>Show answer</b></summary>

On the linear scale everything before 2017 looks flat next to the recent thousands. The log scale shows the **whole history**: a fast rise in the 1960s, a long plateau of roughly 100–170 objects per year during the Cold War and after, a dip in the early 2000s, and then a steep rise. The recent jump comes mainly from **mega-constellations** of small communication satellites (above all SpaceX's Starlink, launched in batches of dozens from 2019), plus many small Earth-observation and CubeSat missions. For EO, this means far more satellites (and data) than ever, but also more crowded orbits and more space debris.

</details>

---
# 7. Masks: selecting pixels by their values

A comparison returns a **boolean array** (`True`/`False`) of the same shape. Booleans count as 1 and 0, so `.sum()` counts pixels and `.mean()` gives the fraction.

In [ ]:
ndvi = np.array([[0.10, 0.20, 0.35, 0.70],
                 [0.05, 0.45, 0.62, 0.80],
                 [-0.2, 0.30, 0.55, 0.68]])

veg = ndvi > 0.5
print(veg)
print('vegetated pixels:', veg.sum(), '| fraction:', veg.mean().round(2))
print('their NDVI values:', ndvi[veg])           # boolean indexing returns a 1-D array

Combine conditions with `&` (and), `|` (or) and `~` (not), **with brackets** around each condition. Python's own `and`/`or` do not work on arrays:

In [ ]:
moderate = (ndvi > 0.3) & (ndvi <= 0.6)
print(moderate.astype(int))

try:
    ndvi > 0.3 and ndvi <= 0.6
except ValueError as e:
    print('\nValueError:', e)

## 7.1 Classifying: `np.where`, `np.select` and `np.isin`

- `np.where(condition, value_if_true, value_if_false)`: two classes, or replace values where a condition holds.
- `np.select(list_of_conditions, list_of_values, default)`: several classes. Conditions are checked **in order**, and the first true one wins.
- `np.isin(array, list_of_values)`: is each pixel one of these codes? Ideal for **categorical** layers such as land cover or the Sentinel-2 scene classification (SCL).

In [ ]:
water_flag = np.where(ndvi < 0, 1, 0)

veg_class = np.select(
    [ndvi < 0, ndvi < 0.3, ndvi < 0.6],     # checked in this order
    [0,        1,          2],              # 0 water, 1 sparse, 2 moderate
    default=3)                               # 3 dense: everything else
print(veg_class)

scl = np.array([[4, 4, 9, 8],
                [5, 3, 4, 6],
                [4, 10, 4, 4]])               # SCL codes: 3 shadow, 4 veg, 5 bare/built, 6 water, 8-10 cloud/cirrus
clear = np.isin(scl, [4, 5, 6])
print(clear)

**Question:** In the `np.select` call, why must `ndvi < 0` come before `ndvi < 0.3`? What happens if you reverse the order?

<details><summary><b>Show answer</b></summary>

Because the first true condition wins. A pixel with NDVI −0.2 satisfies **both** `ndvi < 0` and `ndvi < 0.3`. If `ndvi < 0.3` came first, water pixels would be labelled 1 (sparse) and class 0 would never appear. Order conditions from most specific to most general, or make them non-overlapping, e.g. `(ndvi >= 0) & (ndvi < 0.3)`.

</details>

## 7.2 Missing data: NaN and NaN-aware statistics

To exclude clouds, set them to `NaN` (float arrays only, section 3.3). Normal statistics then return `NaN`; the **`np.nan...`** versions ignore missing values:

In [ ]:
ndvi_masked = np.where(clear, ndvi, np.nan)
print(ndvi_masked)

print('np.mean:          ', np.mean(ndvi_masked))
print('np.nanmean:       ', np.nanmean(ndvi_masked).round(3))
print('valid pixels:     ', np.isfinite(ndvi_masked).sum(), 'of', ndvi_masked.size)
print('nanmean per row:  ', np.nanmean(ndvi_masked, axis=1).round(3))

**Question:** Why should a summary always report **how many valid pixels** it is based on, not just the mean?

<details><summary><b>Show answer</b></summary>

A mean of 3 clear pixels is far less reliable than a mean of 3,000. If clouds cover most of an area on one date, its mean NDVI may describe only one corner (e.g. a park, not the city around it), and comparing it with another date is misleading. Report the valid count or fraction, and drop dates or zones below a threshold, as we do in section 10 (and as `zonal_mean` did in Week 6).

</details>

## 7.3 Bit-packed quality flags

Landsat's `QA_PIXEL` band (which you used in Earth Engine in Week 6) packs **16 yes/no flags into one 16-bit integer**. Each bit has a meaning: bit 0 fill, 1 dilated cloud, 2 cirrus, 3 cloud, 4 cloud shadow, 5 snow, 6 clear, 7 water; bits 8–15 hold confidence levels (USGS Landsat Collection 2 documentation). To test a bit, shift a 1 to that position (`1 << bit`) and use the **bitwise** `&`:

In [ ]:
qa_pixel = np.array([21824, 21952, 22280], dtype=np.uint16)   # three example QA_PIXEL values

for v in qa_pixel:
    print(v, '=', np.binary_repr(v, width=16), '(bit 15 ... bit 0)')

def bit_set(qa, bit):
    """True where the given bit of a QA array is 1."""
    return (qa & (1 << bit)) != 0

for bit, name in [(1, 'dilated cloud'), (3, 'cloud'), (4, 'cloud shadow'), (6, 'clear'), (7, 'water')]:
    print(f'bit {bit} {name:14s}', bit_set(qa_pixel, bit))

In [ ]:
# One mask for "bad" pixels: dilated cloud OR cloud OR cloud shadow (bits 1, 3, 4), as in Week 6
BAD_BITS = (1 << 1) | (1 << 3) | (1 << 4)
print('bad-bit pattern:', np.binary_repr(BAD_BITS, width=16))
print('bad pixels:     ', (qa_pixel & BAD_BITS) != 0)

**Question:** Reading the decoded flags, what kind of surface is each of the three example pixels? Why can't you simply test `qa_pixel == 21824` to find clear pixels?

<details><summary><b>Show answer</b></summary>

21824: **clear** (bit 6) land. 21952: clear **water** (bits 6 and 7). 22280: **cloud** (bit 3, and bits 8–9 both set = high cloud confidence). Testing for one exact value fails because many *different* values mean "clear": the confidence bits, snow, water and so on vary from pixel to pixel. Testing the individual bits you care about is robust.

</details>

---
# 8. Reshaping

`reshape` changes the shape without changing the values, as long as the total number of elements stays the same. Use `-1` for "whatever length fits".

In [ ]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, -1).shape)      # -1 -> 6

## 8.1 Raster ↔ table

Many analyses (scatter plots of band against band, statistics, machine learning) want a **table**: one row per pixel, one column per band. A `(bands, rows, cols)` image becomes `(pixels, bands)` with one reshape and a transpose, and the result goes back the same way:

In [ ]:
rng = np.random.default_rng(3)
image = rng.uniform(0, 0.5, size=(4, 30, 40))          # 4 bands: blue, green, red, nir

n_bands, n_rows, n_cols = image.shape
table = image.reshape(n_bands, -1).T                   # (4, 1200) -> (1200, 4)
print('table shape:', table.shape)

back = table.T.reshape(n_bands, n_rows, n_cols)        # and back to an image
print('round trip identical:', np.array_equal(back, image))

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(table[:, 2], table[:, 3], s=3, alpha=0.4)   # every pixel: red vs NIR ("feature space")
ax.set_xlabel('Red reflectance')
ax.set_ylabel('NIR reflectance')
plt.show()

**Question:** Why the `.T` (transpose)? What would `image.reshape(-1, 4)` give instead, and why is it wrong?

<details><summary><b>Show answer</b></summary>

`image.reshape(n_bands, -1)` gives `(4, 1200)`: each **row** is one band. `.T` turns it into `(1200, 4)`: one row per pixel. `image.reshape(-1, 4)` also has shape `(1200, 4)`, but NumPy fills it by reading the array in memory order (band 0 row by row first), so each "row" would contain 4 **neighbouring pixels of the blue band**, not the 4 bands of one pixel. The shape looks right, the content is nonsense: always check with a pixel you know, e.g. `table[0]` should equal `image[:, 0, 0]`.

</details>

## 8.2 Block averaging: a coarser resolution without loops

`a[::2, ::2]` takes every second pixel: it **samples**, it does not average. To aggregate 2 × 2 blocks (e.g. 10 m → 20 m), reshape each block into its own pair of axes and average over them:

In [ ]:
fine = np.arange(16, dtype=float).reshape(4, 4)

sampled = fine[::2, ::2]
blocks = fine.reshape(2, 2, 2, 2).mean(axis=(1, 3))   # (rows/2, 2, cols/2, 2) -> mean over each block

print('fine:\n', fine)
print('every 2nd pixel:\n', sampled)
print('2x2 block mean:\n', blocks)

## 8.3 Where is the maximum?

`argmax()` on a 2-D array returns a position in the **flattened** array. `np.unravel_index` converts it back to (row, column):

In [ ]:
lst_map = lst.mean(axis=0)                     # the annual mean LST map from section 6
flat_index = lst_map.argmax()
row, col = np.unravel_index(flat_index, lst_map.shape)
print(f'flat index {flat_index} -> row {row}, col {col}, LST {lst_map[row, col]:.1f} °C')

---
# 9. Broadcasting: combining arrays of different shapes

NumPy can combine arrays of **different shapes** by stretching the smaller one, without copying it in memory. The rules, comparing shapes **from the right**:

1. two axes are compatible if they are **equal** or one of them is **1**;
2. a missing axis on the left counts as 1;
3. axes of length 1 are stretched to match.

| a | b | a + b |
|---|---|---|
| `(12, 5, 6)` | `(5, 6)` | `(12, 5, 6)`: the map is applied to every month |
| `(3, 30, 40)` | `(3, 1, 1)` | `(3, 30, 40)`: one value per band |
| `(3, 30, 40)` | `(3,)` | **error**: 3 is compared with 40 |
| `(5, 1)` | `(1, 8)` | `(5, 8)`: a grid |

`np.newaxis` (or `None`) inserts an axis of length 1 where you need it.

In [ ]:
# 1) Per-band scale and offset: e.g. DN -> reflectance with different factors per band
dn = rng.integers(500, 4000, size=(3, 30, 40)).astype(np.float32)
scale = np.array([0.0001, 0.0001, 0.00002])
offset = np.array([0.0, 0.0, -0.1])

refl = dn * scale[:, np.newaxis, np.newaxis] + offset[:, np.newaxis, np.newaxis]   # (3,1,1) stretches over rows and cols
print('reflectance shape:', refl.shape)

try:
    dn * scale                    # (3, 30, 40) * (3,) -> 3 is compared with 40
except ValueError as e:
    print('ValueError:', e)

In [ ]:
# 2) Anomalies: each month minus the pixel's long-term mean. (12,5,6) - (5,6) just works.
lst_anomaly = lst - lst.mean(axis=0)
print('anomaly shape:', lst_anomaly.shape)
print('mean anomaly of July at each pixel ~ +10 °C:', lst_anomaly[6].mean().round(1))

In [ ]:
# 3) Distance from a point (e.g. a weather station, a tree, a city centre) for every pixel
x = x_ul + (np.arange(200) + 0.5) * res        # 200 columns of 20 m (section 4.1)
y = y_ul - (np.arange(150) + 0.5) * res        # 150 rows
station_x, station_y = 522000, 173500

dist = np.sqrt((x[np.newaxis, :] - station_x) ** 2 + (y[:, np.newaxis] - station_y) ** 2)   # (1,200) and (150,1) -> (150,200)
print('distance grid:', dist.shape)

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(dist, extent=[x[0] - res / 2, x[-1] + res / 2, y[-1] - res / 2, y[0] + res / 2], cmap='viridis')
ax.plot(station_x, station_y, 'r^', ms=10, label='station')
ax.contour(x, y, dist, levels=[500, 1000, 1500], colors='white', linewidths=0.8)
fig.colorbar(im, ax=ax, label='Distance (m)')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.tick_params(axis='x', labelrotation=30)
ax.legend()
plt.show()

## 9.1 Evaluating a model for every parameter combination

Broadcasting also replaces nested loops over **model parameters**. Here, a linear model `y = m·x + c` evaluated for 3 intercepts, 6 slopes and 21 x values in one call. Each input gets its own axis:

In [ ]:
def linear_model(c, m, x):
    """y = m * x + c (works for scalars and for arrays of any broadcastable shapes)."""
    return m * x + c

c = np.linspace(0.0, 1.0, 3)          # intercepts
m = np.linspace(0.0, 0.05, 6)         # slopes
x = np.linspace(0.0, 10.0, 21)        # x values

y_all = linear_model(c[:, np.newaxis, np.newaxis],     # (3, 1, 1)
                     m[np.newaxis, :, np.newaxis],     # (1, 6, 1)
                     x[np.newaxis, np.newaxis, :])     # (1, 1, 21)
print('result shape:', y_all.shape)                    # (3, 6, 21): every c, every m, every x

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i, ax in enumerate(axes):
    for j in range(len(m)):
        ax.plot(x, y_all[i, j], label=f'm = {m[j]:.2f}')
    ax.set_title(f'c = {c[i]:.1f}')
    ax.set_xlabel('x')
axes[0].set_ylabel('y')
axes[-1].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Question:** Predict the result shape (or an error) for: `(4, 1) + (3,)`, `(2, 3, 4) * (3, 1)`, `(2, 3, 4) * (2, 3)`.

<details><summary><b>Show answer</b></summary>

`(4, 1) + (3,)` → `(4, 3)`. `(2, 3, 4) * (3, 1)` → `(2, 3, 4)` (the 3s match, the 1 stretches to 4, and the missing left axis to 2). `(2, 3, 4) * (2, 3)` → **error**: from the right, 4 is compared with 3. Fix it with `b[:, :, np.newaxis]`, shape `(2, 3, 1)`.

</details>

---
# 10. Putting it together: a year of Sentinel-2 over Richmond Park

Now a real workflow. We read **red**, **NIR** and the **scene classification (SCL)** for every reasonably clear Sentinel-2 date in 2024, for an area of about 3.6 × 3.5 km covering Richmond Park and the surrounding town (Roehampton, Putney, Kingston). Reading uses `rioxarray` exactly as in Week 6; **everything after that is plain NumPy**, written as small functions, as Part B requires.

If the STAC catalogue cannot be reached, the same code runs on a **simulated** stack with the same structure, so you can still follow every step (the printout tells you which one you have).

In [ ]:
AOI = (-0.280, 51.430, -0.230, 51.460)     # (min_lon, min_lat, max_lon, max_lat): Richmond Park and surroundings
YEAR = 2024
MAX_TILE_CLOUD = 30                        # % cloud over the whole 110 km tile (local clouds are masked later)


def find_scenes(bbox, year, max_cloud, tile='30UXC', max_scenes=16):
    """Search Earth Search for Sentinel-2 L2A scenes of one tile; keep the clearest scene per date.

    Returns a list of pystac Items sorted by date (at most max_scenes, spread over the year).
    """
    from pystac_client import Client
    catalog = Client.open('https://earth-search.aws.element84.com/v1')
    search = catalog.search(collections=['sentinel-2-l2a'], bbox=bbox,
                            datetime=f'{year}-01-01/{year}-12-31',
                            query={'eo:cloud_cover': {'lt': max_cloud}}, max_items=200)
    best = {}
    for item in search.items():
        if f'_{tile}_' not in item.id:                 # one tile only, so every date has the same pixel grid
            continue
        date = item.datetime.date()
        if date not in best or item.properties['eo:cloud_cover'] < best[date].properties['eo:cloud_cover']:
            best[date] = item
    items = [best[d] for d in sorted(best)]
    if len(items) > max_scenes:                        # spread the selection evenly over the year
        keep = np.linspace(0, len(items) - 1, max_scenes).round().astype(int)
        items = [items[k] for k in keep]
    return items


def read_window(href, bbox, overview_level=None):
    """Read a lon/lat window of a cloud-optimised GeoTIFF and return it as a 2-D NumPy array (stored values)."""
    import rioxarray as rxr
    from rasterio.warp import transform_bounds
    da = rxr.open_rasterio(href, overview_level=overview_level).squeeze('band', drop=True)
    xmin, ymin, xmax, ymax = transform_bounds('EPSG:4326', da.rio.crs, *bbox)
    return da.rio.clip_box(minx=xmin, miny=ymin, maxx=xmax, maxy=ymax).values


def scale_info(item, asset):
    """Scale, offset and nodata for one asset; skips the offset if Earth Search already applied it (Week 6)."""
    rb = item.assets[asset].extra_fields.get('raster:bands', [{}])[0]
    scale, offset, nodata = rb.get('scale', 1e-4), rb.get('offset', 0.0), rb.get('nodata', 0)
    if item.properties.get('earthsearch:boa_offset_applied'):
        offset = 0.0
    return scale, offset, nodata


def load_scenes(items, bbox):
    """Read red, NIR (both at 20 m: first overview) and SCL (native 20 m) for every item.

    Returns a list of dicts with NumPy arrays and metadata.
    """
    scenes = []
    for k, item in enumerate(items, start=1):
        red = read_window(item.assets['red'].href, bbox, overview_level=0)
        nir = read_window(item.assets['nir'].href, bbox, overview_level=0)
        scl = read_window(item.assets['scl'].href, bbox)
        n_r, n_c = min(a.shape[0] for a in (red, nir, scl)), min(a.shape[1] for a in (red, nir, scl))
        scenes.append({'date': np.datetime64(item.datetime.date()),
                       'red': red[:n_r, :n_c], 'nir': nir[:n_r, :n_c], 'scl': scl[:n_r, :n_c],
                       'red_scale': scale_info(item, 'red'), 'nir_scale': scale_info(item, 'nir')})
        print(f'  {k:2d}/{len(items)} {item.datetime:%Y-%m-%d}  shape {red.shape}  tile cloud {item.properties["eo:cloud_cover"]:.0f} %')
    return scenes

In [ ]:
def simulate_scenes(n_dates=16, shape=(167, 174), seed=2024):
    """A simulated year of Sentinel-2 red/NIR/SCL with the same structure as load_scenes() (fallback)."""
    rng = np.random.default_rng(seed)
    rows, cols = np.mgrid[0:shape[0], 0:shape[1]]
    park = ((rows - 95) / 70) ** 2 + ((cols - 80) / 60) ** 2 < 1                       # a large park...
    wood = park & (np.sin(rows / 9) * np.cos(cols / 11) > 0.35)                         # ...with woodland patches
    pond = ((rows - 110) / 6) ** 2 + ((cols - 95) / 14) ** 2 < 1
    doy = np.linspace(20, 345, n_dates).round().astype(int)
    scenes = []
    for d in doy:
        season = np.exp(-((d - 160) / 70) ** 2)                                         # summer peak
        grass_ndvi = 0.35 + 0.40 * np.exp(-((d - 130) / 55) ** 2)                       # greens early, dries in summer
        ndvi = np.where(park, grass_ndvi, 0.18 + 0.08 * season)                         # town
        ndvi = np.where(wood, 0.35 + 0.50 * season, ndvi)                               # deciduous trees
        ndvi = ndvi + rng.normal(0, 0.03, shape)
        nir = 0.28 + 0.12 * ndvi
        red = nir * (1 - ndvi) / (1 + ndvi)
        red, nir = np.where(pond, 0.03, red), np.where(pond, 0.02, nir)
        scl = np.where(pond, 6, np.where(ndvi > 0.3, 4, 5))
        cloud = np.zeros(shape, bool)                                                    # a few random clouds
        for _ in range(rng.integers(0, 5)):
            r0, c0, rad = rng.integers(0, shape[0]), rng.integers(0, shape[1]), rng.integers(15, 75)
            cloud |= (rows - r0) ** 2 + (cols - c0) ** 2 < rad ** 2
        red, nir = np.where(cloud, 0.35, red), np.where(cloud, 0.38, nir)
        scl = np.where(cloud, 9, scl)
        to_dn = lambda r: np.clip(r * 10000, 1, 65535).astype(np.uint16)
        scenes.append({'date': np.datetime64(f'{YEAR}-01-01') + np.timedelta64(int(d) - 1, 'D'),
                       'red': to_dn(red), 'nir': to_dn(nir), 'scl': scl.astype(np.uint8),
                       'red_scale': (1e-4, 0.0, 0), 'nir_scale': (1e-4, 0.0, 0)})
    return scenes


try:
    items = find_scenes(AOI, YEAR, MAX_TILE_CLOUD)
    print(f'{len(items)} dates found. Reading the windows (about 1-2 minutes)...')
    scenes = load_scenes(items, AOI)
    REAL_DATA = len(scenes) >= 4
except Exception as e:
    print(f'Could not read Sentinel-2 ({type(e).__name__}: {e}).')
    REAL_DATA = False

if not REAL_DATA:
    print('Using the SIMULATED stack instead: same structure, invented values.')
    scenes = simulate_scenes()
print('scenes:', len(scenes), '| DATA:', 'REAL Sentinel-2' if REAL_DATA else 'SIMULATED')

## 10.1 From stored numbers to an NDVI stack

Each step is a small function that takes arrays and returns arrays. Note the order: **convert to float and handle nodata first** (section 3), then mask clouds, then calculate.

In [ ]:
CLEAR_CLASSES = [4, 5, 6, 11]       # SCL: vegetation, bare/built, water, snow. Excludes clouds, shadows, defective


def dn_to_reflectance(dn, scale, offset, nodata=0):
    """Stored integers -> float32 reflectance, with nodata set to NaN."""
    refl = dn.astype(np.float32) * scale + offset
    return np.where(dn == nodata, np.nan, refl)


def calc_ndvi(red, nir):
    """NDVI = (NIR - red) / (NIR + red); NaN where the denominator is 0."""
    with np.errstate(divide='ignore', invalid='ignore'):
        return (nir - red) / (nir + red)


def clear_mask(scl, clear_classes=CLEAR_CLASSES):
    """True where the scene classification is one of the clear classes."""
    return np.isin(scl, clear_classes)


def scene_ndvi(scene):
    """Cloud-masked NDVI (2-D array) for one scene dict."""
    red = dn_to_reflectance(scene['red'], *scene['red_scale'])
    nir = dn_to_reflectance(scene['nir'], *scene['nir_scale'])
    return np.where(clear_mask(scene['scl']), calc_ndvi(red, nir), np.nan)


ndvi_stack = np.stack([scene_ndvi(s) for s in scenes], axis=0)     # list of 2-D arrays -> (time, rows, cols)
dates = np.array([s['date'] for s in scenes])

print('stack:', ndvi_stack.shape, ndvi_stack.dtype, f'{ndvi_stack.nbytes / 1e6:.1f} MB')
print('dates:', dates[0], '...', dates[-1])
print(f'NDVI range (valid pixels): {np.nanmin(ndvi_stack):.2f} to {np.nanmax(ndvi_stack):.2f}')

**Question:** Why do we read red and NIR at `overview_level=0` (20 m) rather than at their native 10 m?

<details><summary><b>Show answer</b></summary>

Because SCL, our cloud mask, only exists at **20 m**. Reading the 10 m bands at their first overview (20 m) puts all three layers on the **same pixel grid**, so the arrays line up element by element and NumPy masking is a one-liner. The alternative is to upsample SCL to 10 m, e.g. `np.repeat(np.repeat(scl, 2, axis=0), 2, axis=1)`, which also works but needs care at the edges of the window. Either way: before combining arrays, make sure they have the **same shape and the same grid**.

</details>

## 10.2 Drop the cloudy dates

The tile-level cloud filter (`eo:cloud_cover < 30 %`) describes the whole 110 km tile, not our small window. So we measure the **valid fraction of each date inside our window** (a mean over the two spatial axes) and keep only dates with at least 60 % valid pixels, using a boolean index on the time axis:

In [ ]:
valid_fraction = np.isfinite(ndvi_stack).mean(axis=(1, 2))        # one value per date
for d, f in zip(dates, valid_fraction):
    print(d, f'{f:5.0%}', '' if f >= 0.6 else '  <- dropped')

keep = valid_fraction >= 0.6
ndvi_stack, dates = ndvi_stack[keep], dates[keep]
print('\nkept:', ndvi_stack.shape)

## 10.3 Time series and per-pixel summaries

The same stack, reduced along different axes: over **space** for a time series, over **time** for maps.

In [ ]:
mean_ts = np.nanmean(ndvi_stack, axis=(1, 2))
p10_ts, p90_ts = np.nanpercentile(ndvi_stack, [10, 90], axis=(1, 2))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.fill_between(dates, p10_ts, p90_ts, alpha=0.25, label='10th–90th percentile of pixels')
ax.plot(dates, mean_ts, marker='o', label='mean')
ax.set_ylabel('NDVI (–)')
ax.set_title(f"NDVI through {YEAR}, Richmond Park area ({'Sentinel-2' if REAL_DATA else 'SIMULATED'})")
ax.legend()
plt.show()

In [ ]:
def temporal_summary(stack):
    """Per-pixel summaries of a (time, rows, cols) stack that may contain NaN.

    Returns a dict of 2-D arrays: median, amplitude (max - min) and the index of the date of maximum.
    """
    all_nan = np.all(np.isnan(stack), axis=0)                 # pixels never observed clear
    filled = np.where(np.isnan(stack), -np.inf, stack)        # so argmax ignores NaN
    peak_index = np.where(all_nan, -1, filled.argmax(axis=0))
    return {'median': np.nanmedian(stack, axis=0),
            'amplitude': np.nanmax(stack, axis=0) - np.nanmin(stack, axis=0),
            'peak_index': peak_index}


summary = temporal_summary(ndvi_stack)
date_months = dates.astype('datetime64[M]').astype(int) % 12 + 1             # month number (1-12) of each date
peak_month = np.where(summary['peak_index'] >= 0, date_months[summary['peak_index']], np.nan)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
panels = [(summary['median'], 'Median NDVI', 'RdYlGn', (-0.1, 0.9)),
          (summary['amplitude'], 'Seasonal amplitude (max − min NDVI)', 'magma', (0, 0.7)),
          (peak_month, 'Month of peak NDVI', plt.get_cmap('viridis', 12), (0.5, 12.5))]
for ax, (arr, title, cmap, (vmin, vmax)) in zip(axes, panels):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

**Question:** Why do we replace NaN with `-np.inf` before `argmax`, and what would go wrong for a pixel that was cloudy on every date?

<details><summary><b>Show answer</b></summary>

`argmax` does not skip NaN (and `np.nanargmax` raises an error if a pixel is NaN on **every** date). `-inf` is smaller than any real NDVI, so it can never be chosen as the maximum. A pixel that was never clear would be all `-inf`, and `argmax` would happily return 0 (the first date), which looks like a real answer. So we flag those pixels with `all_nan` and set them to −1 / NaN: **missing must stay visibly missing**.

</details>

**Question:** Look at the month-of-peak map. Do the open grassland and the woodland peak at the same time? What could explain a difference?

<details><summary><b>Show answer</b></summary>

Check your own map, but a typical pattern in southern England is that **grassland greens up early and peaks in spring** (April–June), then browns in a dry summer, while **deciduous tree canopies peak in mid-summer** (June–August) and drop sharply in autumn. Urban pixels have low NDVI and a weak, noisy seasonal cycle, so their "peak month" is close to random. Before interpreting this, remember the limitations: only the dates that passed the cloud filter count, so a gap in June can shift the peak to May or July; and grass sward management (grazing by deer, mowing) changes the curve.

</details>

## 10.4 A simple rule-based classification and its area

Combine the two summary maps with `np.select`. The thresholds are illustrative: in a real project you would justify them from the histograms or from reference data.

In [ ]:
med, amp = summary['median'], summary['amplitude']
lc = np.select([np.isnan(med),
                med < 0.1,
                med < 0.4,
                amp < 0.3],
               [0, 1, 2, 3],
               default=4)
names = ['no data', 'water / bare', 'built-up / sparse', 'green, stable', 'green, strongly seasonal']
colours = ['white', '#2b83ba', '#bababa', '#1a9641', '#a6d96a']

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(lc, cmap=ListedColormap(colours), vmin=-0.5, vmax=4.5, interpolation='nearest')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=n) for c, n in zip(colours, names)],
          loc='upper left', bbox_to_anchor=(1, 1))
ax.set_title('Rule-based classes from median NDVI and amplitude')
ax.axis('off')
plt.show()

PIXEL_AREA_HA = 20 * 20 / 10_000                    # one 20 m pixel = 0.04 ha
classes, counts = np.unique(lc, return_counts=True)
for k, n in zip(classes, counts):
    print(f'{names[k]:26s} {n:6d} pixels  {n * PIXEL_AREA_HA:7.1f} ha  {n / lc.size:6.1%}')

## 10.5 Save the results

`np.save` stores one array (`.npy`); `np.savez_compressed` stores several named arrays in one compressed file (`.npz`), which is a convenient way to keep intermediate results in Part B so you don't have to download everything again. (For results that others will open in GIS software, save a GeoTIFF with its coordinates, as in Week 6.)

In [ ]:
out = DATA_DIR / 'processed' / f'richmond_ndvi_{YEAR}.npz'
np.savez_compressed(out, ndvi_stack=ndvi_stack, dates=dates.astype(str),
                    median=summary['median'], amplitude=summary['amplitude'], classes=lc)
print(f'saved {out} ({out.stat().st_size / 1e6:.1f} MB)')

with np.load(out) as f:
    print('arrays in the file:', f.files)
    print('stack read back:', f['ndvi_stack'].shape, '| first date:', f['dates'][0])

**Question:** Which of the functions in this section could you reuse unchanged in Part B, for example for the London urban heat island topic? What would you need to change?

<details><summary><b>Show answer</b></summary>

`dn_to_reflectance`, `calc_ndvi`, `clear_mask`, `temporal_summary` and the valid-fraction filter work for **any** stack of arrays: they don't depend on Richmond Park, on Sentinel-2's tile or on notebook variables. You would change the inputs (the AOI, years, bands: e.g. B11 and B8 for NDBI), and for Landsat temperature you would swap `clear_mask` for a QA-bit mask (section 7.3) and `dn_to_reflectance` for the thermal scale and offset. This is exactly the point of writing functions with clear arguments and return values: the second use is almost free.

</details>

---
# 11. Common NumPy errors and what they mean

| Message (or symptom) | Usual cause | Fix |
|---|---|---|
| `operands could not be broadcast together with shapes (3,30,40) (3,)` | shapes don't align from the right | add axes: `b[:, np.newaxis, np.newaxis]` |
| `IndexError: index 6 is out of bounds for axis 0 with size 6` | indices start at 0; the last is `n - 1` | check `.shape`; use `-1` for the last |
| `The truth value of an array with more than one element is ambiguous` | `and`/`or`/`if` used on an array | use `&`, `\|`, `~` with brackets, or `.any()` / `.all()` |
| `ValueError: cannot convert float NaN to integer` | NaN put into an integer array | `.astype(np.float32)` first |
| `RuntimeWarning: Mean of empty slice` | a pixel or date with only NaN | expected for fully cloudy pixels; check the valid count |
| `cannot reshape array of size 1200 into shape (7, 100)` | sizes must match exactly | check `.size`; use `-1` for one axis |
| Values like 65336 or NDVI of 40 (no error!) | unsigned-integer overflow | convert to float before arithmetic |
| Original raster changed after editing a "subset" (no error!) | the subset was a view | `.copy()` |
| Session crashed: out of RAM | arrays too big (`float64`, full tiles) | windows, overviews, `float32` |

## Using AI help sensibly

Colab's Gemini and other assistants are good at explaining error messages and shapes. Following the module's AI rules, use them to **understand**, not to replace your reasoning. Useful prompts:

> *I am learning NumPy. Explain this error message in simple terms, and show a 2-line example that causes and fixes it. Don't rewrite my code.*

> *My array has shape (12, 167, 174) = (time, rows, cols). Explain what `np.nanmean(a, axis=(1, 2))` returns, with a tiny example.*

Always run and **check shapes** of AI-suggested code: a common AI mistake is to mix up band-first and band-last order, or rows and columns.

---
# Summary

| Topic | Key points |
|---|---|
| Why arrays | one dtype, contiguous memory, vectorised arithmetic: tens to hundreds of times faster than pixel loops |
| Anatomy | always print `.shape`, `.dtype`; memory = size × itemsize; prefer `float32` |
| dtype traps | uint16 wraps below 0; ints truncate; ints have no NaN → convert to float, nodata → NaN |
| Creating | `zeros`/`full_like`, `arange` (integers), `linspace` (decimals), `default_rng(seed)`, `meshgrid` for coordinates |
| Indexing | `a[band, row, col]`; slices keep axes, integers drop them; arrays of rows/cols sample points; slices are **views** |
| Axes | the axis you name disappears: `axis=0` of a time stack → map; `axis=(1, 2)` → time series; `keepdims` |
| Masks | comparisons → booleans; `&`, `\|`, `~`; `np.where`, `np.select`, `np.isin`; `np.nan...` statistics; bits with `&` and `<<` |
| Reshape | `(bands, rows, cols)` ↔ `(pixels, bands)` with `reshape(n_bands, -1).T`; block means with reshape + mean; `unravel_index` |
| Broadcasting | align shapes from the right; 1 stretches; `np.newaxis` adds the axis you need |
| Saving | `np.save` / `np.savez_compressed` and `np.load` for intermediate arrays; GeoTIFF for maps |

# Resources and practice

| Resource | What it is |
|---|---|
| [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) | Official beginner tutorial |
| [NumPy broadcasting guide](https://numpy.org/doc/stable/user/basics.broadcasting.html) | The rules, with diagrams |
| [NumPy cheat sheet (DataCamp)](https://www.datacamp.com/cheat-sheet/numpy-cheat-sheet-data-analysis-in-python) | One-page reference for creating, indexing and manipulating arrays |
| [Python Data Science Handbook, ch. 2](https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html) | Free book chapter: arrays, ufuncs, aggregation, broadcasting, masks |
| [Earth Lab: intro to NumPy arrays](https://www.earthdatascience.org/courses/intro-to-earth-data-science/scientific-data-structures-python/numpy-arrays/) | NumPy taught with Earth-science data |
| [From Python to NumPy (N. Rougier)](https://www.labri.fr/perso/nrougier/from-python-to-numpy/) | Free book on thinking in vectorised code (more advanced) |
| [100 NumPy exercises](https://github.com/rougier/numpy-100) | Graded exercises **with solutions**, from beginner to hard: good practice for the Thursday session |
| [W3Schools NumPy exercises](https://www.w3schools.com/python/numpy/numpy_exercises.asp) | Short browser-based quiz questions, checked instantly |
| [NumPy Learn](https://numpy.org/learn/) | More tutorials, books and videos |
| [Landsat Collection 2 QA bands (USGS)](https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands) | The bit definitions used in section 7.3 |